![Snowflake](https://www.snowflake.com/wp-content/themes/flavor/assets/img/logo-snowflake-sans-word.svg)
# CredibanCo — Hands-On Lab
**RFP 10010806** · Plataforma de Datos · Snowflake 2026

---

# Track 2 — Ingeniería de Datos
**Rol:** CRB_DATA_ANALYTICS | **Tiempo:** 15 min | **Criterio:** Pipeline declarativo, calidad, observabilidad, CI/CD, linaje

In [ ]:
USE ROLE CRB_DATA_ANALYTICS;
USE DATABASE CREDIBANCO_HOL;
USE WAREHOUSE CREDIBANCO_HOL_WH;

## Lanza este prompt en CoCo AHORA
Copia el bloque de abajo y pégalo en Cortex Code **antes de continuar**. CoCo ejecuta todo en paralelo mientras tú avanzas con el resto del notebook.

In [ ]:
-- PROMPT PARA COCO: copiar TODO este bloque y pegar en Cortex Code
--
-- Necesito que hagas estas dos tareas en paralelo. Paraleliza todo lo que puedas.
--
-- TAREA 1: Conectar a la API de la Superfinanciera de Colombia
-- - Crea Network Rule + External Access Integration para datos.gov.co
-- - Crea una UDF Python que consulte https://www.datos.gov.co/resource/mcec-87by.json
-- - Crea la tabla CREDIBANCO_HOL.PLATAFORMA.TRM_HISTORICA y carga 365 dias de TRM
-- - Programa un Task diario a las 8AM Colombia (CRON '0 13 * * *' UTC)
--
-- TAREA 2: Ingestar datos desde S3 publico a Snowflake
-- - Bucket: s3://sfquickstarts/summit_dev_day_2026_automated_intelligence_hol/
-- - Archivo: parquet/customers.parquet (2M filas)
-- - Crea external stage, infiere schema, crea tabla y carga con COPY INTO
-- - Tabla destino: CREDIBANCO_HOL.PLATAFORMA.CUSTOMERS_S3
--   ^^^ CAMBIA ESTE NOMBRE SI QUIERES (ej: MI_TABLA_CLIENTES, DATOS_S3, etc.)
--
-- Al terminar muestra: conteo de filas de ambas tablas y 5 filas de ejemplo de cada una.
-- Usa la conexion credibanco-hol. No pidas confirmacion, ejecuta todo de una vez.

---
**Mientras CoCo trabaja, continúa ejecutando los bloques de abajo.**

---

## Bloque 1 — Evidencia: Pipeline declarativo sin Spark

In [ ]:
-- Dynamic Tables: pipeline bronce→plata→oro SIN orquestador
SHOW DYNAMIC TABLES IN DATABASE CREDIBANCO_HOL;

Verificamos que la capa **Silver** (datos limpios y enriquecidos) se genera automáticamente desde la capa Bronze. Sin Jobs de Spark, sin orquestador.

In [ ]:
-- Ver datos transformados en la capa Silver
SELECT COUNT(*) AS filas_silver FROM CREDIBANCO_HOL.PAGOS.DT_AUTORIZACIONES_SILVER_USER;
SELECT * FROM CREDIBANCO_HOL.PAGOS.DT_AUTORIZACIONES_SILVER_USER LIMIT 5;

El linaje nativo traza la cadena completa de transformación: desde la tabla fuente hasta las capas Silver y Gold — sin metadata externa.

In [ ]:
-- Linaje nativo: trazar la cadena end-to-end
SELECT * FROM TABLE(SNOWFLAKE.CORE.GET_LINEAGE(
  'CREDIBANCO_HOL.PAGOS.DT_AUTORIZACIONES_SILVER_USER', 'table', 'upstream', 3
));

## Bloque 2 — Ejecutar: Crear nueva DT + ver propagación

Las Dynamic Tables mantienen agregados actualizados automáticamente. Cada 5 minutos, Snowflake recalcula sin necesidad de Tasks manuales ni orquestadores.

In [ ]:
-- Crear una Dynamic Table nueva: agregados por hora
CREATE OR REPLACE DYNAMIC TABLE CREDIBANCO_HOL.PAGOS.DT_HOURLY_USER
  TARGET_LAG = '5 minutes'
  WAREHOUSE = CREDIBANCO_HOL_WH
AS
SELECT DATE_TRUNC('hour', FECHA_HORA) AS hora,
       CIUDAD, COUNT(*) AS num_tx, SUM(MONTO) AS monto_total
FROM CREDIBANCO_HOL.PAGOS.AUTORIZACIONES
GROUP BY 1, 2;

Verificamos que la Dynamic Table se refresca automáticamente. El `REFRESH_STATE` muestra si está al día o procesando cambios.

In [ ]:
-- Verificar refresh automático
SHOW DYNAMIC TABLES LIKE 'DT_HOURLY%' IN SCHEMA CREDIBANCO_HOL.PAGOS;

Insertamos un registro nuevo y verificamos que la Dynamic Table lo captura automáticamente en el próximo refresh — **sin intervención manual**.

In [ ]:
-- Insertar dato nuevo y ver propagación
INSERT INTO CREDIBANCO_HOL.PAGOS.AUTORIZACIONES
SELECT 999999, 1, 1, '4111111111111111', '5411', 'Bogota',
       CURRENT_TIMESTAMP(), 9999999, '00', 'ECOMMERCE';

## Bloque 3 — Ingesta desde API externa (TRM Colombia)
Conectamos Snowflake a la API pública de la Superfinanciera para traer la TRM del último año. Todo se ejecuta dentro de Snowflake — sin herramientas externas.

In [ ]:
-- Paso 1: Crear Network Rule para permitir acceso a datos.gov.co
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE NETWORK RULE CREDIBANCO_HOL.PLATAFORMA.NR_DATOS_GOV
  MODE = EGRESS TYPE = HOST_PORT
  VALUE_LIST = ('www.datos.gov.co:443');

Creamos la integración de acceso externo que autoriza a Snowflake a conectarse a la API. Sin esta integración, el sandbox es hermético.

In [ ]:
-- Paso 2: External Access Integration
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION EAI_DATOS_GOV
  ALLOWED_NETWORK_RULES = (CREDIBANCO_HOL.PLATAFORMA.NR_DATOS_GOV)
  ENABLED = TRUE;

La **UDF Python** encapsula la lógica de llamada a la API. Se ejecuta dentro de Snowflake — los datos nunca salen de la plataforma. Retorna la TRM histórica como tabla SQL.

In [ ]:
-- Paso 3: UDF Python que consulta la API y retorna la TRM
CREATE OR REPLACE FUNCTION CREDIBANCO_HOL.PLATAFORMA.GET_TRM_HISTORICA(dias INT)
  RETURNS TABLE (fecha DATE, trm FLOAT)
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.11'
  PACKAGES = ('requests')
  EXTERNAL_ACCESS_INTEGRATIONS = (EAI_DATOS_GOV)
  HANDLER = 'TRMHandler'
AS $$
import requests
from datetime import date
class TRMHandler:
    def process(self, dias):
        url = f'https://www.datos.gov.co/resource/mcec-87by.json?$limit={dias}&$order=vigenciadesde DESC'
        r = requests.get(url)
        for row in r.json():
            yield (date.fromisoformat(row['vigenciadesde'][:10]), float(row['valor']))
$$;

Ejecutamos la UDF para traer los últimos 30 días de TRM directamente desde la Superfinanciera. Datos **reales y en vivo** — no mockups.

In [ ]:
-- Paso 4: Consultar la TRM del último año (datos REALES de la Superfinanciera)
SELECT * FROM TABLE(CREDIBANCO_HOL.PLATAFORMA.GET_TRM_HISTORICA(30))
ORDER BY fecha DESC;

Persistimos la TRM en una tabla Snowflake. Esto permite consultas históricas sin volver a llamar la API — y se puede automatizar con un Task diario.

In [ ]:
-- Paso 5: Almacenar en tabla persistente
CREATE OR REPLACE TABLE CREDIBANCO_HOL.PLATAFORMA.TRM_HISTORICA AS
SELECT * FROM TABLE(CREDIBANCO_HOL.PLATAFORMA.GET_TRM_HISTORICA(365));

Verificamos la carga: cantidad de días, rango de fechas y TRM promedio. Todo en una sola query SQL.

In [ ]:
-- Paso 6: Verificar datos cargados
SELECT COUNT(*) AS dias, MIN(fecha) AS desde, MAX(fecha) AS hasta,
       ROUND(AVG(trm), 2) AS trm_promedio
FROM CREDIBANCO_HOL.PLATAFORMA.TRM_HISTORICA;

### Prompt CoCo (opcional)
Si quieres que CoCo además programe la actualización diaria, copia este prompt:

> **Crea un Task llamado TASK_TRM_DIARIA que ejecute la función GET_TRM_HISTORICA todos los días a las 8AM Colombia (UTC-5) y haga MERGE INTO TRM_HISTORICA para agregar solo los días nuevos. Usa CRON '0 13 * * *' (8AM COT = 1PM UTC).**

In [ ]:
-- Verificación final
SELECT 'T2_COMPLETO' AS status,
  (SELECT COUNT(*) FROM CREDIBANCO_HOL.PAGOS.DT_HOURLY_USER) AS filas_nueva_dt;